# Fine-tuning your RAG Chatbot LLM with Databricks Mosaic AI

## Fine Tuning or RAG?
RAG and Instruction fine-tuning work together! If you have a relevant RAG use-case, you can start with RAG leveraging a foundational model, and then specialize your model if your corpus is specific to your business (ex: not part of the foundation model training) or need specific behavior (ex: answer to a specific task such as entity extraction)

Start with RAG on a Foundation Model, evaluate how your model is working and where it can be improved, and build a Fine Tuned dataset accordingly!


## Fine Tuning a Chatbot on Databricks Documentation 

In this demo, we will fine tune Mistral (or llama) on Databricks Documentation for a RAG chatbot to help a customer answer Databricks-related questions. 

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-0.png?raw=true" width="1200px">

Databricks provides a simple, built-in API to fine tune the model and evaluate its performance. Let's get started!

Documentation for Fine Tuning on Databricks:
- Overview page: [Databricks Fine-tuning APIs](https://docs.databricks.com/en/large-language-models/foundation-model-training/index.html)
- SDK Guide: [Setting up a fine-tuning session with Fine-tuning APIs](https://docs.databricks.com/en/large-language-models/foundation-model-training/create-fine-tune-run.html)


In [0]:
# Let's start by installing our libraries
%pip install --quiet databricks-genai==1.1.4 mlflow==2.16.2
%pip install --quiet databricks-sdk==0.40.0
dbutils.library.restartPython()

In [0]:
%run ../_resources/00-setup

## Fine Tuning Dataset

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-1.png?raw=true" width="700px" style="float: right">

Building a high quality fine tuning dataset is key for improving your model's performance. 

The training dataset needs to match the data you will be sending to your final model. <br/>
If you have a RAG application, you need to fine tune with the full RAG instruction pipeline so that your model can learn how to extract the proper information from the context and answer the way you want.

For this demo, we have loaded a fine tuning dataset for you containing the following:

* A Databricks user question (ex: how do I start a Warehouse?)
* A page or chunk from Databricks documentation relevant to this question
* The expected answer, reviewed by a human

## A note on Databricks Mosaic AI Evaluation Framework

Databricks makes it easy to build, deploy and review RAG application. Using Databricks review application, it's easy to build your Fine Tuning Dataset.

To see how it's done, install the `dbdemos.install('llm-rag-chatbot')` demo!

In [0]:
training_dataset = spark.sql("""
  SELECT q.id as question_id, q.question, a.answer, d.url, d.content FROM training_dataset_question q
      INNER JOIN databricks_documentation d on q.doc_id = d.id
      INNER JOIN training_dataset_answer   a on a.question_id = q.id 
    WHERE answer IS NOT NULL""")
display(training_dataset)

## Preparing the Dataset for Chat Completion

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-2.png?raw=true" width="700px" style="float: right">

Because we're fine tuning a chatbot, we need to prepare our dataset for **Chat completion**.

Chat completion requires a list of **role** and **prompt**, following the OpenAI standard. This standard has the benefit of transforming our input into a prompt following our LLM instruction pattern. <br/>
Note that each foundation model might be trained with a different instruction type, so it's best to use the same type when fine tuning.<br/>
*We recommend using Chat Completion whenever possible.*

```
[
  {"role": "system", "content": "[system prompt]"},
  {"role": "user", "content": "Here is a documentation page:[RAG context]. Based on this, answer the following question: [user question]"},
  {"role": "assistant", "content": "[answer]"}
]
```

*Remember that your Fine Tuning dataset should be the same format as the one you're using for your RAG application.<br/>*

#### Training Data Type

Databricks supports a large variety of dataset formats (Volume files, Delta tables, and public Hugging Face datasets in .jsonl format), but we recommend preparing the dataset as Delta tables within your Catalog as part of a proper data pipeline to ensure production quality.
*Remember, this step is critical and you need to make sure your training dataset is of high quality.*

Let's create a small pandas UDF to help create our final chat completion dataset.<br/>

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

base_model_name = "meta-llama/Llama-3.2-3B-Instruct"

system_prompt = """You are a highly knowledgeable and professional Databricks Support Agent. Your goal is to assist users with their questions and issues related to Databricks. Answer questions as precisely and accurately as possible, providing clear and concise information. If you do not know the answer, respond with "I don't know." Be polite and professional in your responses. Provide accurate and detailed information related to Databricks. If the question is unclear, ask for clarification.\n"""

@pandas_udf("array<struct<role:string, content:string>>")
def create_conversation(content: pd.Series, question: pd.Series, answer: pd.Series) -> pd.Series:
    def build_message(c,q,a):
        user_input = f"Here is a documentation page that could be relevant: {c}. Based on this, answer the following question: {q}"
        if "mistral" in base_model_name:
            #Mistral doesn't support system prompt
            return [
                {"role": "user", "content": f"{system_prompt} \n{user_input}"},
                {"role": "assistant", "content": a}]
        else:
            return [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": a}]
    return pd.Series([build_message(c,q,a) for c, q, a in zip(content, question, answer)])


training_data, eval_data = training_dataset.randomSplit([0.9, 0.1], seed=42)

training_data.select(create_conversation("content", "question", "answer").alias('messages')).write.mode('overwrite').saveAsTable("chat_completion_training_dataset")
eval_data.write.mode('overwrite').saveAsTable("chat_completion_evaluation_dataset")

display(spark.table('chat_completion_training_dataset'))

In [0]:
spark.sql("create or replace view chat_completion_evaluation_view as select * from chat_completion_training_dataset limit 50")


## Starting a Fine Tuning Run

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-3.png?raw=true" width="700px" style="float: right">

Once the training is done, your model will automatically be saved within Unity Catalog and available for you to serve!


#### 1.1) Instruction Fine Tune our Baseline Model

In this demo, we'll be using the API on the table we just created to programatically fine tune our LLM.

However, you can also create a new Fine Tuning experiment from the UI!

In [0]:
from databricks.model_training import foundation_model as fm
#Return the current cluster id to use to read the dataset and send it to the fine tuning cluster. See https://docs.databricks.com/en/large-language-models/foundation-model-training/create-fine-tune-run.html#cluster-id
def get_current_cluster_id():
  import json
  return json.loads(dbutils.notebook.entry_point.getDbutils().notebook().getContext().safeToJson())['attributes']['clusterId']


#Let's clean the model name
registered_model_name = f"{catalog}.{db}." + re.sub(r'[^a-zA-Z0-9]', '_',  base_model_name)



run = fm.create(
    data_prep_cluster_id=get_current_cluster_id(),  # required if you are using delta tables as training data source. This is the cluster id that we want to use for our data prep job.
    model=base_model_name,  # Here we define what model we used as our baseline
    train_data_path=f"{catalog}.{db}.chat_completion_training_dataset",
    task_type="CHAT_COMPLETION",  # Change task_type="INSTRUCTION_FINETUNE" if you are using the fine-tuning API for completion.
    register_to=registered_model_name,
    training_duration="5ep", #only 5 epoch to accelerate the demo. Check the mlflow experiment metrics to see if you should increase this number
    learning_rate="5e-7",
)

print(run)

#### 1.2) Tracking Fine Tuning Runs via the MLFlow Experiment

To monitor the progress of an ongoing or past fine tuning run, you can open the run from the MLFlow Experiment. Here you will find valuable information on how you may wish to tweak future runs to get better results. For example:
* Adding more epochs if you see your model still improving at the end of your run
* Increasing learning rate if loss is decreasing, but very slowly
* Decreasing learning rate if loss is fluctuating widely

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-experiment.png?raw=true" width="1200px">

In [0]:
displayHTML(f'Open the <a href="/ml/experiments/{run.experiment_id}/runs/{run.run_id}/model-metrics">training run on MLFlow</a> to track the metrics')
#Track the run details
display(run.get_events())

#helper function waiting on the run to finish - see the _resources folder for more details
wait_for_run_to_finish(run)

#### 1.3) Deploy Fine Tuned Model to a Serving Endpoint

<img src="https://docs.databricks.com/en/_images/create-provisioned-throughput-ui.png" width="600px" style="float: right">

Once ready, the model will be available in Unity Catalog.

From here, you can use the UI to deploy your model, or you can use the API. For reproducibility, we'll be using the API below:


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedEntityInput,
    EndpointCoreConfigInput,
    AiGatewayConfig,
    AiGatewayInferenceTableConfig
)

serving_endpoint_name = "dbdemos_llm_fine_tuned_llama3p2_3B"
w = WorkspaceClient()

# Create the AI Gateway configuration
ai_gateway_config = AiGatewayConfig(
    inference_table_config=AiGatewayInferenceTableConfig(
        enabled=True,
        catalog_name=catalog,
        schema_name=db,
        table_name_prefix="fine_tuned_llm_inference"
    )
)

endpoint_config = EndpointCoreConfigInput(
    name=serving_endpoint_name,
    served_entities=[
        ServedEntityInput(
            entity_name=registered_model_name,
            entity_version=get_latest_model_version(registered_model_name),
            min_provisioned_throughput=0,
            max_provisioned_throughput=100,
            scale_to_zero_enabled=True
        )
    ]
)

force_update = False
existing_endpoint = next(
    (e for e in w.serving_endpoints.list() if e.name == serving_endpoint_name), None
)

if existing_endpoint is None:
    print(f"Creating the endpoint {serving_endpoint_name}, this will take a few minutes to package and deploy the endpoint...")
    w.serving_endpoints.create_and_wait(name=serving_endpoint_name, config=endpoint_config, ai_gateway=ai_gateway_config)
else:
    print(f"Endpoint {serving_endpoint_name} already exists...")
    if force_update:
        w.serving_endpoints.update_config_and_wait(
            served_entities=endpoint_config.served_entities,
            name=serving_endpoint_name
        )


#### 1.4) Testing the Model Endpoint

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/llm-fine-tuning/databricks-llm-fine-tuning-4.png?raw=true" width="600px" style="float: right">

That's it! We're now ready to serve our Fine Tuned model and start asking questions!

The reponses will now be improved and specialized from the Databricks documentation and our RAG chatbot formatted output!

In [0]:
import mlflow
from mlflow import deployments
#remove the answer to get only the system + user role.
test_dataset = spark.table('chat_completion_training_dataset').selectExpr("slice(messages, 1, size(messages)-1) as messages").limit(1)
#Get the first messages
messages = test_dataset.toPandas().iloc[0].to_dict()['messages'].tolist()

client = mlflow.deployments.get_deploy_client("databricks")
client.predict(endpoint=serving_endpoint_name, inputs={"messages": messages, "max_tokens": 100})


## Next Step: Evaluating our Fine Tuned Model

This is looking good! Fine tuning our model was just a simple API call. But how can we measure the improvement compared to the baseline model?

Databricks makes this easy too! In the next section, we'll leverage MLFlow Evaluate capabilities to compare the fine tune model against the baseline Foundation Model to see how successful our tuning run was.

Open the [02.2-llm-evaluation]($./02.2-llm-evaluation) notebook to benchmark our new custom LLM!